In [1]:
import pandas as pd
import joblib
import sys
from pathlib import Path
from sklearn import set_config
from sklearn.model_selection import TunedThresholdClassifierCV
from sklearn.metrics import make_scorer

set_config(enable_metadata_routing=True)

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import precision_score, recall_score, fbeta_score

# Works whether kernel cwd is project root or notebooks/
cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    src_path = cwd / "src"
elif (cwd.parent / "src").exists():
    src_path = cwd.parent / "src"
else:
    raise RuntimeError("Could not find src/ directory")

sys.path.insert(0, str(src_path))

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)


In [2]:
RUN_TAG = "v1"
ARTIFACT_DIR = Path("../artifacts/model_search") / RUN_TAG

SUMMARY_PATH = ARTIFACT_DIR / "cv_summary.csv"
SEARCH_RESULTS_PATH = ARTIFACT_DIR / "search_results.joblib"

summary_df = pd.read_csv(SUMMARY_PATH)
search_results = joblib.load(SEARCH_RESULTS_PATH)

summary_df.head()

,model,best_score_refit_f2,best_mean_test_recall,best_mean_test_precision,best_mean_test_f2,best_mean_test_pr_auc,best_params
0,svc_rbf,0.916118,0.944924,0.816951,0.916118,0.921851,"{'clf__C': np.float64(72.86653737491046), 'clf__gamma': np.float64(0.026619018884890575)}"
1,rf,0.900404,0.925349,0.813167,0.900404,0.923146,"{'clf__max_depth': 10, 'clf__n_estimators': 220}"
2,knn,0.899316,0.922166,0.818403,0.899316,0.915006,"{'clf__n_neighbors': 30, 'clf__weights': 'distance'}"
3,hgb,0.889750,0.898952,0.855415,0.889750,0.931531,"{'clf__learning_rate': np.float64(0.052466345336252856), 'clf__max_leaf_nodes': 21}"
4,logreg,0.873978,0.919436,0.729929,0.873978,0.810868,{'clf__C': np.float64(4.5705630998014515)}


In [3]:
KOI_train_set = pd.read_csv("../data/processed/KOI_train_set.csv")
KOI_test_set = pd.read_csv("../data/processed/KOI_test_set.csv")
K2P_set = pd.read_csv("../data/processed/K2P_set.csv")

In [4]:
X_train = KOI_train_set.drop(columns=["label", "group_id"])
y_train = KOI_train_set["label"]
group_id = KOI_train_set["group_id"]

In [5]:
X_test = KOI_test_set.drop(columns=["label", "group_id"])
y_test = KOI_test_set["label"]

In [6]:
X_k2p = K2P_set.drop(columns=["label", "group_id"])
y_k2p = K2P_set["label"]

We will use the tuned models from `search_results.joblib` as a candidate pool for 3 model profiles:
* Best F2 model
* Best Recall model (with a minimum precision floor)
* Best Precision model (with a minimum recall floor)


To find them, we will try to tune the thresholds for each model to find the best candidates.

In [7]:
MIN_PRECISION = 0.5
MIN_RECALL = 0.5


def recall_with_precision_floor(y_true, y_pred):
    precision = precision_score(y_true, y_pred, zero_division=0)
    if precision < MIN_PRECISION:
        return 0.0
    return recall_score(y_true, y_pred, zero_division=0)


def precision_with_recall_floor(y_true, y_pred):
    recall = recall_score(y_true, y_pred, zero_division=0)
    if recall < MIN_RECALL:
        return 0.0
    return precision_score(y_true, y_pred, zero_division=0)


scoring_profiles = {
    "best_f2": make_scorer(fbeta_score, beta=2, zero_division=0),
    "best_recall_pmin_0_5": make_scorer(recall_with_precision_floor),
    "best_precision_rmin_0_5": make_scorer(precision_with_recall_floor),
}

candidate_model_names = list(search_results.keys())
candidate_model_names

['logreg', 'svc_rbf', 'knn', 'rf', 'hgb']

In [8]:
threshold_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
tuned_threshold_models = {}
threshold_rows = []

for model_name in candidate_model_names:
    base_estimator = search_results[model_name].best_estimator_
    tuned_threshold_models[model_name] = {}

    for profile_name, profile_scorer in scoring_profiles.items():
        tuned = TunedThresholdClassifierCV(
            estimator=base_estimator,
            scoring=profile_scorer,
            response_method="auto",
            thresholds=100,
            cv=threshold_cv,
            n_jobs=-1,
            refit=True,
            random_state=42,
            store_cv_results=True,
        )
        tuned.fit(X_train, y_train, groups=group_id)

        tuned_threshold_models[model_name][profile_name] = tuned
        threshold_rows.append(
            {
                "model": model_name,
                "profile": profile_name,
                "best_threshold": tuned.best_threshold_,
                "best_score": tuned.best_score_,
            }
        )
        print(f"fitted: {model_name} | {profile_name}")

threshold_tuning_summary = pd.DataFrame(threshold_rows).sort_values(
    ["model", "profile"],
    ascending=[True, True],
).reset_index(drop=True)

threshold_tuning_summary

fitted: logreg | best_f2
fitted: logreg | best_recall_pmin_0_5
fitted: logreg | best_precision_rmin_0_5
fitted: svc_rbf | best_f2
fitted: svc_rbf | best_recall_pmin_0_5
fitted: svc_rbf | best_precision_rmin_0_5
fitted: knn | best_f2
fitted: knn | best_recall_pmin_0_5
fitted: knn | best_precision_rmin_0_5
fitted: rf | best_f2
fitted: rf | best_recall_pmin_0_5
fitted: rf | best_precision_rmin_0_5
fitted: hgb | best_f2
fitted: hgb | best_recall_pmin_0_5
fitted: hgb | best_precision_rmin_0_5


,model,profile,best_threshold,best_score
0,hgb,best_f2,0.131340,0.922793
1,hgb,best_precision_rmin_0_5,0.885432,0.946760
2,hgb,best_recall_pmin_0_5,0.032118,0.992710
3,knn,best_f2,0.363636,0.915164
4,knn,best_precision_rmin_0_5,0.878788,0.934325
5,knn,best_recall_pmin_0_5,0.030303,0.995904
6,logreg,best_f2,0.343406,0.882803
7,logreg,best_precision_rmin_0_5,0.818113,0.851883
8,logreg,best_recall_pmin_0_5,0.050501,0.991346
9,rf,best_f2,0.271035,0.913233
